# Fixed propagation delay

This notebook demonstrates the same ideal, matched transmission line in the time and frequency domains. Circulax keeps the delay explicit rather than replacing it with a rational approximation:

$$b_1(t)=a\,a_2(t-\tau), \qquad b_2(t)=a\,a_1(t-\tau).$$

`TransmissionLine` expresses this directly in its component physics with `past = signals.at_delay(tau)`. In transient analysis this produces a literal time shift. In AC analysis it produces the corresponding phase factor $\exp(-j2\pi f\tau)$. The analyses below therefore test the same component physics rather than separate models.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from circulax import compile_circuit
from circulax.components.electronic import Resistor, SmoothPulse, TransmissionLine, VoltageSourceAC

jax.config.update("jax_enable_x64", True)
plt.rcParams.update({"figure.figsize": (8, 3.5), "axes.grid": True})

## Transient: watch the edge move

A smooth step drives a matched line. Because the load equals $Z_0$, there is no reflection and the output should be an attenuated copy of the input shifted by $\tau$. Constant DC prehistory means no artificial edge appears at startup.

In [ ]:
tau = 1.0e-9
z0 = 50.0
attenuation = 0.9
source_delay = 2.0e-9
rise_time = 0.8e-9

models = {
    "source": SmoothPulse,
    "line": TransmissionLine,
    "resistor": Resistor,
    "ground": lambda: 0,
}

transient_netlist = {
    "instances": {
        "GND": {"component": "ground"},
        "VIN": {
            "component": "source",
            "settings": {"V": 1.0, "delay": source_delay, "tr": rise_time},
        },
        "TL": {
            "component": "line",
            "settings": {"tau": tau, "z0": z0, "attenuation": attenuation},
        },
        "RL": {"component": "resistor", "settings": {"R": z0}},
    },
    "connections": {
        "GND,p1": ("VIN,p2", "RL,p2"),
        "VIN,p1": "TL,p1",
        "TL,p2": "RL,p1",
    },
    "ports": {"input": "TL,p1", "output": "TL,p2"},
}

transient_circuit = compile_circuit(transient_netlist, models)

In [ ]:
sample_times = jnp.linspace(0.0, 6.0e-9, 601)
solution = transient_circuit.transient(
    t0=0.0,
    t1=float(sample_times[-1]),
    dt0=1.0e-11,
    saveat=sample_times,
    max_steps=4000,
    throw=True,
)

v_in = transient_circuit.port(solution.ys, "input")
v_out = transient_circuit.port(solution.ys, "output")
analytic_out = attenuation * jax.nn.sigmoid(
    10.0 * (sample_times - source_delay - tau) / rise_time
)
max_transient_error = float(jnp.max(jnp.abs(v_out - analytic_out)))
print(f"Maximum transient error: {max_transient_error:.2e}")
assert max_transient_error < 2e-4

In [ ]:
fig, ax = plt.subplots()
ax.plot(sample_times * 1e9, v_in, label="input", lw=2)
ax.plot(sample_times * 1e9, v_out, label="delayed output", lw=2)
ax.plot(sample_times * 1e9, analytic_out, "k--", label="analytic shift", lw=1.5)
ax.annotate(
    rf"$\tau={tau * 1e9:.1f}\,\mathrm{{ns}}$",
    xy=(source_delay * 1e9 + tau * 1e9 / 2, 0.5),
    ha="center",
)
ax.set(xlabel="Time (ns)", ylabel="Voltage (V)", title="A matched line delays the waveform without reshaping it")
ax.legend()
plt.show()

## AC: read the same delay from phase

For a matched line, $S_{21}=a\exp(-j2\pi f\tau)$. Its magnitude is constant and its unwrapped phase is linear, so the group delay is

$$-\frac{1}{2\pi}\frac{d\angle S_{21}}{df}=\tau.$$

In [ ]:
# Very large shunts register both external nodes while changing S by less than 1e-11.
ac_netlist = {
    "instances": {
        "GND": {"component": "ground"},
        "TL": {
            "component": "line",
            "settings": {"tau": tau, "z0": z0, "attenuation": attenuation},
        },
        "R1": {"component": "resistor", "settings": {"R": 1e15}},
        "R2": {"component": "resistor", "settings": {"R": 1e15}},
    },
    "connections": {
        "GND,p1": ("R1,p2", "R2,p2"),
        "TL,p1": "R1,p1",
        "TL,p2": "R2,p1",
    },
    "ports": {"port1": "TL,p1", "port2": "TL,p2"},
}

ac_circuit = compile_circuit(ac_netlist, models)
frequencies = jnp.linspace(1e6, 2e9, 501)
scattering = ac_circuit.sp(ports=["port1", "port2"], freqs=frequencies, z0=z0)
s21 = scattering[:, 1, 0]
analytic_s21 = attenuation * jnp.exp(-2j * jnp.pi * frequencies * tau)
max_ac_error = float(jnp.max(jnp.abs(s21 - analytic_s21)))
print(f"Maximum AC error: {max_ac_error:.2e}")
assert max_ac_error < 1e-9

In [ ]:
phase = np.unwrap(np.angle(np.asarray(s21)))
group_delay = -np.gradient(phase, np.asarray(frequencies)) / (2 * np.pi)

fig, (ax_mag, ax_delay) = plt.subplots(1, 2, figsize=(10, 3.5))
ax_mag.plot(np.asarray(frequencies) / 1e9, 20 * np.log10(np.abs(np.asarray(s21))), lw=2)
ax_mag.axhline(20 * np.log10(attenuation), color="k", ls="--", label="analytic")
ax_mag.set(xlabel="Frequency (GHz)", ylabel=r"$|S_{21}|$ (dB)", title="Constant attenuation")
ax_mag.legend()

ax_delay.plot(np.asarray(frequencies) / 1e9, group_delay * 1e9, lw=2)
ax_delay.axhline(tau * 1e9, color="k", ls="--", label=rf"$\tau={tau * 1e9:.1f}$ ns")
ax_delay.set(xlabel="Frequency (GHz)", ylabel="Group delay (ns)", title="Delay recovered from phase")
ax_delay.legend()
plt.tight_layout()
plt.show()

## Harmonic balance: rotate each harmonic

HB represents a periodic waveform by harmonics of a fundamental frequency $f_0$. The solver applies the delay to harmonic $k$ as

$$X_k(t-\tau)=X_k(t)\exp(-j2\pi kf_0\tau).$$

Here a sinusoidal source drives the same matched line through a $50\,\Omega$ source resistance. We compare the fundamental output/input ratio with the same analytical factor used for AC.

In [ ]:
fundamental = 250e6
hb_netlist = {
    "instances": {
        "GND": {"component": "ground"},
        "VS": {"component": "source_ac", "settings": {"V": 1.0, "freq": fundamental}},
        "RS": {"component": "resistor", "settings": {"R": z0}},
        "TL": {
            "component": "line",
            "settings": {"tau": tau, "z0": z0, "attenuation": attenuation},
        },
        "RL": {"component": "resistor", "settings": {"R": z0}},
    },
    "connections": {
        "GND,p1": ("VS,p2", "RL,p2"),
        "VS,p1": "RS,p1",
        "RS,p2": "TL,p1",
        "TL,p2": "RL,p1",
    },
    "ports": {"line_input": "TL,p1", "line_output": "TL,p2"},
}

hb_models = {**models, "source_ac": VoltageSourceAC}
hb_circuit = compile_circuit(hb_netlist, hb_models, backend="dense")
hb_time, hb_spectrum = hb_circuit.hb(
    freq=fundamental,
    harmonics=5,
    rtol=1e-9,
    atol=1e-9,
    max_steps=30,
)

input_spectrum = hb_circuit.port(hb_spectrum, "line_input")
output_spectrum = hb_circuit.port(hb_spectrum, "line_output")
hb_ratio = output_spectrum[1] / input_spectrum[1]
analytic_ratio = attenuation * jnp.exp(-2j * jnp.pi * fundamental * tau)
hb_error = float(jnp.abs(hb_ratio - analytic_ratio))
higher_harmonics = float(jnp.max(jnp.abs(output_spectrum[2:])))

print(f"HB fundamental ratio: {hb_ratio:.6f}")
print(f"Analytical ratio:      {analytic_ratio:.6f}")
print(f"Fundamental error:     {hb_error:.2e}")
print(f"Largest higher harmonic: {higher_harmonics:.2e}")
assert hb_error < 2e-7
assert higher_harmonics < 2e-8

In [ ]:
hb_times = np.arange(hb_time.shape[0]) / (hb_time.shape[0] * fundamental)
hb_input = np.asarray(hb_circuit.port(hb_time, "line_input"))
hb_output = np.asarray(hb_circuit.port(hb_time, "line_output"))
expected_hb_output = 0.5 * attenuation * np.sin(2 * np.pi * fundamental * (hb_times - tau))

fig, ax = plt.subplots()
ax.plot(hb_times * 1e9, hb_input, "o-", label="line input")
ax.plot(hb_times * 1e9, hb_output, "o-", label="HB delayed output")
ax.plot(hb_times * 1e9, expected_hb_output, "k--", label="analytic shift")
ax.set(xlabel="Time within one period (ns)", ylabel="Voltage (V)", title="HB periodic steady state")
ax.legend()
plt.show()

Transient, AC, and HB all recover the configured **1 ns** propagation delay. Because `TransmissionLine` requests one fixed delay through `signals.at_delay(tau)`, each solver infers and applies the appropriate time- or frequency-domain representation automatically.